# 🕵️ Lab W5-2 — Data Leakage และ Temporal Validation

**รายวิชาระบบสนับสนุนการตัดสินใจ · สัปดาห์ที่ 5 — Data Mining I**

Lab นี้ใช้คู่กับสื่อจำลอง **Leakage Hunter** (`/sims/leakage-hunter`)
ตัวเลขที่คุณคำนวณได้ในสมุดเล่มนี้ต้องตรงกับตัวเลขบนหน้าจอสื่อจำลองทุกหลัก

## สิ่งที่จะได้เรียนรู้
1. ตรวจจับ **data leakage** ได้ก่อนที่โมเดลจะขึ้นใช้งานจริง
2. อธิบายว่าเหตุใด **การแบ่งข้อมูลแบบสุ่มจึงปกปิดปัญหา** ที่การแบ่งตามเวลาเปิดเผย
3. วัด **calibration** ไม่ใช่แค่ AUC และอธิบายว่าเหตุใดจึงสำคัญกว่าในงานสินเชื่อ
4. เขียน **checklist ตรวจการรั่ว** ที่นำไปใช้กับโปรเจกต์อื่นได้

## ข้อมูล
`loan_leaky.csv` — คำขอสินเชื่อ 12,000 รายการ ปี 2023–2025

ตารางนี้ถูกดึงจากระบบปฏิบัติการ **ณ วันนี้** จึงมีคอลัมน์ที่บันทึกเหตุการณ์
ซึ่งเกิดขึ้น **หลัง** การอนุมัติปนอยู่ด้วย

In [ ]:
import numpy as np
import pandas as pd

pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

URL = ("https://raw.githubusercontent.com/babankbro/ksu-dss-course/"
       "master/datasets/week05/loan_leaky.csv")
df = pd.read_csv(URL)
df["status_bad"] = (df.account_status == "ค้างชำระ").astype(int)

print(f"จำนวนคำขอ   : {len(df):,}")
print(f"ช่วงเวลา    : {df.application_date.min()} ถึง {df.application_date.max()}")
print(f"อัตราผิดนัด : {df.defaulted.mean()*100:.2f}%")
print(f"\nคอลัมน์ทั้งหมด:\n  " + "\n  ".join(df.columns))

## ส่วนที่ 1 — ตรวจจับการรั่วโดยยังไม่ต้องฝึกโมเดล

วิธีที่เร็วที่สุดคือดู **AUC ของตัวแปรเดี่ยว** เทียบกับคำตอบ
ตัวแปรใดที่เพียงตัวเดียวก็แยกได้เกือบสมบูรณ์ แทบจะรับประกันได้ว่ารั่ว

In [ ]:
def auc(scores, y) -> float:
    """AUC ด้วยวิธี Mann–Whitney U รองรับค่าที่ซ้ำกันด้วยอันดับเฉลี่ย"""
    r = pd.Series(np.asarray(scores, dtype=float)).rank()
    y = np.asarray(y)
    n_pos = int(y.sum())
    n_neg = len(y) - n_pos
    return (r[y == 1].sum() - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg)


LEGIT = ["income_at_application", "debt_ratio_at_application",
         "credit_history_months_at_application", "age_at_application",
         "prev_loans_at_application", "loan_amount_at_application"]
SUSPECT = ["collection_calls", "days_since_last_payment", "status_bad"]

### 🧑‍💻 งานที่ 1
คำนวณ AUC ของทุกตัวแปรเดี่ยว แล้วเรียงจากค่าที่ห่างจาก 0.50 มากที่สุด
จากนั้นระบุว่าตัวแปรใดน่าสงสัยว่ารั่ว พร้อมเหตุผลเชิงเวลา (ไม่ใช่เชิงสถิติ)

*เฉลยที่ถูกต้อง: ตัวแปรที่รั่วทั้ง 3 ตัวได้ AUC = 1.0000 พอดี
ส่วนตัวแปรที่ถูกต้องอยู่ในช่วง 0.42–0.63*

In [ ]:
# เขียนโค้ดของคุณที่นี่


> **หลักการ** ถ้าตัวแปรเดี่ยวให้ AUC เกิน 0.90 ในปัญหาที่ยากโดยธรรมชาติ
> ให้ถือว่า **รั่วไว้ก่อน** จนกว่าจะพิสูจน์ได้ว่าไม่รั่ว
> ไม่ใช่ถือว่าเป็นตัวแปรที่ดีจนกว่าจะพิสูจน์ได้ว่ารั่ว

## ส่วนที่ 2 — ฝึกโมเดลทั้ง 4 กรณี

ใช้ logistic regression แบบ gradient descent ที่เขียนเอง
เพื่อให้ผลตรงกับสื่อจำลองทุกทศนิยม (สื่อจำลองใช้อัลกอริทึมเดียวกันนี้ในเบราว์เซอร์)

In [ ]:
def fit_logistic(X, y, epochs=400, lr=0.5):
    """gradient descent เต็มชุด เริ่มจากน้ำหนักศูนย์ — ผลเหมือนกันทุกครั้ง"""
    Xb = np.c_[np.ones(len(X)), X]
    w = np.zeros(Xb.shape[1])
    for _ in range(epochs):
        p = 1 / (1 + np.exp(-Xb @ w))
        w -= lr * (Xb.T @ (p - y)) / len(y)
    return w


def predict(X, w):
    return 1 / (1 + np.exp(-(np.c_[np.ones(len(X)), X] @ w)))

### 🧑‍💻 งานที่ 2
เขียนฟังก์ชัน `experiment(cols, split)` ที่

* `split="random"` → สลับลำดับด้วย `random_state=42` แล้วแบ่ง 70/30
* `split="temporal"` → เรียงตาม `application_date` แล้วตัด 70% แรกเป็นชุดฝึก
* มาตรฐานฟีเจอร์ด้วยค่าเฉลี่ยและส่วนเบี่ยงเบนของ **ชุดฝึกเท่านั้น**
* คืน AUC ของชุดฝึกและชุดทดสอบ · อัตราผิดนัดที่ทำนาย · อัตราที่เกิดขึ้นจริง

แล้วรันทั้ง 4 กรณี (มี/ไม่มีตัวแปรรั่ว × สุ่ม/ตามเวลา)

*เฉลยที่ถูกต้อง: ตัวแปรที่ถูกต้อง + แบ่งตามเวลา → AUC ทดสอบ 0.6864
ทำนาย 8.10% แต่เกิดขึ้นจริง 12.03%*

In [ ]:
# เขียนโค้ดของคุณที่นี่


> **สิ่งที่ต้องสังเกตให้ได้ 3 ข้อ**
>
> 1. **AUC = 1.0000 ทั้งสองแบบเมื่อมีตัวแปรรั่ว** — การแบ่งตามเวลาไม่ได้ช่วยตรวจจับการรั่ว
>    เพราะตัวแปรที่รั่วก็รั่วอยู่ทั้งในชุดฝึกและชุดทดสอบเท่ากัน
> 2. **ไม่มี error ใดเกิดขึ้นเลย** โมเดลรันผ่าน ตัวเลขสวย และผิดทั้งหมด
> 3. AUC ของโมเดลที่ถูกต้องอยู่ราว 0.68–0.69 ซึ่ง **เป็นค่าปกติของปัญหานี้ในอุตสาหกรรม**
>    ไม่ใช่สัญญาณว่าโมเดลไม่ดี

## ส่วนที่ 3 — สิ่งที่การแบ่งตามเวลาเปิดเผย

ถ้าการแบ่งตามเวลาไม่ได้ช่วยตรวจการรั่ว แล้วมันมีไว้ทำไม

### 🧑‍💻 งานที่ 3
เปรียบเทียบ **calibration** (อัตราที่ทำนาย เทียบกับอัตราที่เกิดขึ้นจริง)
ของโมเดลที่ใช้ตัวแปรถูกต้อง ระหว่างการแบ่งสองแบบ

แล้วตอบว่าการแบ่งแบบใดกำลังโกหก และโกหกเรื่องอะไร

In [ ]:
# เขียนโค้ดของคุณที่นี่


## ส่วนที่ 4 — ต้นตอคือ population drift

### 🧑‍💻 งานที่ 4
แสดงอัตราผิดนัดชำระรายครึ่งปี แล้วอธิบายว่าเหตุใดโมเดลที่ฝึกจากอดีต
จึงประเมินความเสี่ยงของอนาคตต่ำกว่าจริง **เสมอ** ไม่ใช่บางครั้ง

*เฉลยที่ถูกต้อง: อัตราผิดนัดเพิ่มจาก 6.58% เป็น 12.30% ตลอด 3 ปี*

In [ ]:
# เขียนโค้ดของคุณที่นี่


## ส่วนที่ 5 — เปรียบเทียบกับ scikit-learn

### 🧑‍💻 งานที่ 5
ทำซ้ำกรณี "ตัวแปรถูกต้อง + แบ่งตามเวลา" ด้วย
`sklearn.linear_model.LogisticRegression` แล้วเทียบ AUC กับที่คำนวณเอง

จากนั้นตอบว่า ถ้าใช้ `cross_val_score` แบบ 5-fold ธรรมดา
จะตรวจจับปัญหาที่พบใน Lab นี้ได้หรือไม่ เพราะเหตุใด

In [ ]:
# เขียนโค้ดของคุณที่นี่


## ส่วนที่ 6 — Checklist ที่นำไปใช้ต่อได้

### 🧑‍💻 งานที่ 6 (เขียนเป็นข้อความ)

1. เขียน checklist ตรวจการรั่วอย่างน้อย 6 ข้อ ที่ทีมของคุณจะใช้กับ**ทุกโปรเจกต์**
   แต่ละข้อต้องเป็นคำถามที่ตอบได้ว่าใช่หรือไม่ ไม่ใช่หลักการลอย ๆ
2. ยกตัวอย่างการรั่วอีก 2 แบบที่ไม่ปรากฏใน Lab นี้ พร้อมบริบทธุรกิจ
3. ถ้าคุณเป็นผู้ตรวจสอบโมเดลของทีมอื่น และเขาส่งโมเดลที่ AUC 0.94 มาให้
   คุณจะขอดูอะไรบ้างก่อนอนุมัติ

In [ ]:
# เขียนโค้ดของคุณที่นี่


---
## ✅ เกณฑ์การส่งงาน

| องค์ประกอบ | คะแนน |
|---|:--:|
| งานที่ 1 — ตรวจ AUC ตัวแปรเดี่ยวและให้เหตุผลเชิงเวลา | 3 |
| งานที่ 2 — ทดลองครบ 4 กรณีและได้ตัวเลขตรงเฉลย | 4 |
| งานที่ 3 — วิเคราะห์ calibration และระบุว่าการแบ่งใดโกหก | 4 |
| งานที่ 4 — แสดง drift และอธิบายว่าเหตุใดจึงเอนไปทางเดียว | 3 |
| งานที่ 5 — เทียบ sklearn และอธิบายข้อจำกัดของ 5-fold ธรรมดา | 3 |
| งานที่ 6 — checklist และตัวอย่างการรั่วเพิ่มเติม | 3 |
| **รวม** | **20** |

> 💡 ตัวเลขทุกตัวในสมุดเล่มนี้ต้องตรงกับที่แสดงบนสื่อจำลอง `/sims/leakage-hunter`
> ถ้าไม่ตรง แปลว่ามีขั้นตอนใดขั้นตอนหนึ่งผิด — ให้ย้อนกลับไปตรวจก่อนส่ง